# Praktikum 5 Sequential Pattern Analysis
Dataset: Online Retail. Algoritma: GSP dengan batas panjang dua kejadian pembelian.

Identitas: 5054251036 - Prima Surya Nusantara.

Tujuan: menemukan produk yang dibeli pada invoice terdahulu lalu produk pada invoice berikutnya. Satu pelanggan merupakan satu sequence; satu invoice merupakan satu itemset. Support dihitung per pelanggan, bukan per baris atau invoice.

In [26]:
from pathlib import Path
import math
import pandas as pd
import numpy as np
DATA = Path("Online Retail.xlsx")
raw = pd.read_excel(
    DATA,
    dtype={"InvoiceNo": str, "StockCode": str}
)
print(raw.head().to_string(index=False))
print('Shape:', raw.shape)
raw.info()
assert raw.shape == (541909, 8)


InvoiceNo StockCode                         Description  Quantity         InvoiceDate  UnitPrice  CustomerID        Country
   536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00       2.55     17850.0 United Kingdom
   536365     71053                 WHITE METAL LANTERN         6 2010-12-01 08:26:00       3.39     17850.0 United Kingdom
   536365    84406B      CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00       2.75     17850.0 United Kingdom
   536365    84029G KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00       3.39     17850.0 United Kingdom
   536365    84029E      RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00       3.39     17850.0 United Kingdom
Shape: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str

## Eksplorasi dan pembersihan
CustomerID mengidentifikasi sequence, InvoiceDate mengurutkan invoice, dan StockCode mengidentifikasi produk. Description digunakan sebagai label agar variasi deskripsi tidak memecah satu produk menjadi beberapa item. Produk unik dalam satu invoice hanya dicatat sekali. UnitPrice tidak menjadi syarat tambahan karena tugas berfokus pada keberadaan produk, bukan nilai penjualan.

Jumlah penghapusan pada tabel audit bersifat berurutan sehingga tidak dihitung ganda.

In [27]:
print('Nilai kosong per kolom:')
print(raw.isna().sum().to_string())
def stats(frame):
    return {'baris':len(frame), 'invoice':frame.InvoiceNo.nunique(), 'pelanggan':frame.CustomerID.nunique()}
audit = [{'tahap':'Sebelum', 'dihapus':0, **stats(raw)}]
clean = raw.copy()
filters = [
    ('Description terisi', lambda d: d.Description.notna() & d.Description.fillna('').str.strip().ne('')),
    ('Bukan pembatalan', lambda d: ~d.InvoiceNo.str.upper().str.startswith('C', na=False)),
    ('Quantity positif', lambda d: d.Quantity.gt(0)),
    ('CustomerID terisi', lambda d: d.CustomerID.notna()),
]
for label, condition in filters:
    before = len(clean)
    clean = clean.loc[condition(clean)].copy()
    audit.append({'tahap':label, 'dihapus':before-len(clean), **stats(clean)})
audit = pd.DataFrame(audit)
clean['CustomerID'] = clean.CustomerID.astype('int64')
clean['Description'] = clean.Description.str.strip()
assert clean.InvoiceDate.notna().all()
print(audit.to_string(index=False))
print('Rentang tanggal:', clean.InvoiceDate.min(), 's.d.', clean.InvoiceDate.max())
print('Produk unik:', clean.StockCode.nunique())
print('Duplikat baris bersih:', clean.duplicated().sum())


Nilai kosong per kolom:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
             tahap  dihapus  baris  invoice  pelanggan
           Sebelum        0 541909    25900       4372
Description terisi     1454 540455    24446       4372
  Bukan pembatalan     9288 531167    20610       4339
  Quantity positif      474 530693    20136       4339
 CustomerID terisi   132769 397924    18536       4339
Rentang tanggal: 2010-12-01 08:26:00 s.d. 2011-12-09 12:50:00
Produk unik: 3665
Duplikat baris bersih: 5192


## Pembentukan sequence
Invoice diurutkan berdasarkan waktu. Invoice berbeda milik pelanggan yang sama dengan timestamp persis sama digabung menjadi satu kejadian agar tidak membuat urutan waktu buatan. Itemset tiap invoice tetap tersedia di `invoices`; `sequences` merupakan list pelanggan berisi list itemset.

Baris duplikat tidak mengubah keanggotaan produk karena itemset memakai himpunan. Tidak ada perataan seluruh produk menjadi satu deret yang akan memberi urutan palsu di dalam invoice.

In [28]:
invoices = (clean.groupby(['CustomerID','InvoiceNo'], sort=False)
            .agg(InvoiceDate=('InvoiceDate','min'), items=('StockCode',lambda s: sorted(set(s))))
            .reset_index().sort_values(['CustomerID','InvoiceDate','InvoiceNo']))
events = (invoices.groupby(['CustomerID','InvoiceDate'], sort=True)['items']
          .agg(lambda lists: sorted(set().union(*map(set, lists)))).reset_index())
customer_sequences = events.groupby('CustomerID', sort=True)['items'].agg(list)
sequences = customer_sequences.tolist()
N = len(sequences)
lengths = customer_sequences.map(len)
print('Pelanggan / sequence:', N)
print('Invoice:', len(invoices), '| kejadian waktu unik:', len(events))
print('Invoice pada timestamp sama yang digabung:', len(invoices)-len(events))
print('Pelanggan dengan >=2 kejadian:', int((lengths>=2).sum()))
print('Distribusi panjang sequence:')
print(lengths.describe().to_string())
print('Contoh 3 sequence pertama (maksimal 2 kejadian dan 8 item untuk tampilan):')
for cid, seq in customer_sequences.head(3).items():
    print(cid, [items[:8] for items in seq[:2]])


Pelanggan / sequence: 4339
Invoice: 18536 | kejadian waktu unik: 18414
Invoice pada timestamp sama yang digabung: 122
Pelanggan dengan >=2 kejadian: 2841
Distribusi panjang sequence:
count    4339.000000
mean        4.243835
std         7.620524
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       210.000000
Contoh 3 sequence pertama (maksimal 2 kejadian dan 8 item untuk tampilan):
12346 [['23166']]
12347 [['20780', '20782', '21064', '21171', '21731', '22195', '22212', '22375'], ['20719', '20966', '21035', '21041', '21154', '21731', '21832', '21976']]
12348 [['21211', '21213', '21725', '21977', '21981', '21982', '22616', '22951'], ['21726', '21967', '21980', '21983', '21985', 'POST']]


## GSP dengan ruang pencarian terbatas

In [29]:
MIN_SUPPORT = 0.01
min_count = math.ceil(MIN_SUPPORT * N)
singleton_counts = {}
for seq in sequences:
    for item in set().union(*map(set, seq)):
        singleton_counts[item] = singleton_counts.get(item, 0) + 1
frequent_items = sorted(k for k,v in singleton_counts.items() if v >= min_count)
index = {item:i for i,item in enumerate(frequent_items)}
pair_counts = np.zeros((len(index),len(index)), dtype=np.int32)
for seq in sequences:
    first, last = {}, {}
    for t, basket in enumerate(seq):
        for item in basket:
            if item in index:
                j = index[item]
                first.setdefault(j,t)
                last[j] = t
    ids = np.array(sorted(first), dtype=int)
    if len(ids):
        start = np.array([first[j] for j in ids])
        end = np.array([last[j] for j in ids])
        pair_counts[np.ix_(ids,ids)] += start[:,None] < end[None,:]
labels = (clean.groupby('StockCode')['Description']
          .agg(lambda s:s.value_counts().index[0]).to_dict())
rows=[]
for i,j in np.argwhere(pair_counts >= min_count):
    a,b = frequent_items[i], frequent_items[j]
    count = int(pair_counts[i,j])
    rows.append({'A':a, 'B':b, 'produk_A':labels[a], 'produk_B':labels[b],
                 'pelanggan_AB':count, 'pelanggan_A':singleton_counts[a],
                 'support':count/N, 'confidence':count/singleton_counts[a]})
patterns = pd.DataFrame(rows).sort_values(['pelanggan_AB','A','B'], ascending=[False,True,True]).reset_index(drop=True)
singletons = pd.DataFrame([{'StockCode':k,'produk':labels[k],'pelanggan':singleton_counts[k],
                           'support':singleton_counts[k]/N} for k in frequent_items])
print('Ambang absolut:',min_count)
print('L1:',len(frequent_items),'| kandidat C2:',len(index)**2,'| pola L2:',len(patterns))
print('20 pola temporal teratas:')
print(patterns.head(20).to_string(index=False, float_format=lambda x:f'{x:.4f}'))
print('Singleton teratas (confidence tidak didefinisikan untuk pola tanpa antecedent):')
print(singletons.nlargest(10,'pelanggan').to_string(index=False))


Ambang absolut: 44
L1: 1738 | kandidat C2: 3020644 | pola L2: 16215
20 pola temporal teratas:
     A      B                           produk_A                           produk_B  pelanggan_AB  pelanggan_A  support  confidence
85123A 85123A WHITE HANGING HEART T-LIGHT HOLDER WHITE HANGING HEART T-LIGHT HOLDER           407          856   0.0938      0.4755
 22423  22423           REGENCY CAKESTAND 3 TIER           REGENCY CAKESTAND 3 TIER           330          881   0.0761      0.3746
85099B 85099B            JUMBO BAG RED RETROSPOT            JUMBO BAG RED RETROSPOT           312          635   0.0719      0.4913
 84879  84879      ASSORTED COLOUR BIRD ORNAMENT      ASSORTED COLOUR BIRD ORNAMENT           291          678   0.0671      0.4292
 47566  47566                      PARTY BUNTING                      PARTY BUNTING           280          708   0.0645      0.3955
 20725  20725            LUNCH BAG RED RETROSPOT            LUNCH BAG RED RETROSPOT           276          532   0

## Validasi dan sensitivitas
Tiga pola teratas diverifikasi lagi dengan pemindaian subsequence secara langsung. Uji contoh membedakan pembelian bersamaan, pengulangan pada invoice berikutnya, dan urutan terbalik. Perubahan ambang berikut menggunakan matriks dukungan yang sama; pada ambang >=1%, seluruh singleton yang dibutuhkan sudah tercakup.

In [30]:
def contains_pair(seq, a, b):
    seen = False
    for basket in seq:
        if seen and b in basket:
            return True
        if a in basket:
            seen = True
    return False
assert not contains_pair([['A','B']], 'A','B')
assert contains_pair([['A'],['B']], 'A','B')
assert not contains_pair([['B'],['A']], 'A','B')
assert contains_pair([['A'],['A']], 'A','A')
for row in patterns.head(3).itertuples():
    direct = sum(contains_pair(seq,row.A,row.B) for seq in sequences)
    assert direct == row.pelanggan_AB
assert (patterns.confidence.between(0,1)).all()
assert (patterns.support >= MIN_SUPPORT).all()
print('Uji semantik dan verifikasi langsung 3 pola teratas: LULUS')
sensitivity = pd.DataFrame([{'min_support':s,'ambang_pelanggan':math.ceil(s*N),
                             'jumlah_pola_temporal':int((pair_counts>=math.ceil(s*N)).sum())}
                            for s in [0.01,0.02,0.03,0.05]])
print(sensitivity.to_string(index=False))


Uji semantik dan verifikasi langsung 3 pola teratas: LULUS
 min_support  ambang_pelanggan  jumlah_pola_temporal
        0.01                44                 16215
        0.02                87                  1553
        0.03               131                   307
        0.05               217                    11


## Interpretasi tiga pola dengan support tertinggi
Peringkat memakai support, dengan kode produk sebagai pemecah seri. Angka confidence merupakan proporsi historis dalam periode pengamatan; bukan probabilitas kausal atau jaminan pembelian berikutnya. B dapat muncul pada transaksi mana pun setelah A, bukan harus invoice tepat berikutnya.

In [31]:
for rank,row in enumerate(patterns.head(3).itertuples(),1):
    print(f'{rank}. {row.produk_A} ({row.A}) -> {row.produk_B} ({row.B})')
    print(f'   {row.pelanggan_AB}/{N} pelanggan; support {row.support:.2%}; confidence {row.confidence:.2%} ({row.pelanggan_AB}/{row.pelanggan_A}).')
    if row.A == row.B:
        print('   Interpretasi: pembelian ulang produk pada waktu transaksi berbeda. Dapat menjadi dasar pengingat restock kepada pelanggan yang pernah membeli produk tersebut.')
    else:
        print('   Interpretasi: produk B dibeli setelah A oleh kelompok pelanggan tersebut. Dapat diuji sebagai rekomendasi pascapembelian A melalui eksperimen kampanye.')
print('Keterbatasan: observasi satu periode, pelanggan tanpa ID dihapus, pelanggan satu transaksi tetap dalam penyebut, tanpa holdout temporal, dan pola dibatasi dua kejadian singleton.')
print('Kesimpulan: pola menggambarkan transisi historis dan pembelian ulang. Rekomendasi bisnis perlu diuji pada periode berikutnya sebelum digunakan secara operasional.')


1. WHITE HANGING HEART T-LIGHT HOLDER (85123A) -> WHITE HANGING HEART T-LIGHT HOLDER (85123A)
   407/4339 pelanggan; support 9.38%; confidence 47.55% (407/856).
   Interpretasi: pembelian ulang produk pada waktu transaksi berbeda. Dapat menjadi dasar pengingat restock kepada pelanggan yang pernah membeli produk tersebut.
2. REGENCY CAKESTAND 3 TIER (22423) -> REGENCY CAKESTAND 3 TIER (22423)
   330/4339 pelanggan; support 7.61%; confidence 37.46% (330/881).
   Interpretasi: pembelian ulang produk pada waktu transaksi berbeda. Dapat menjadi dasar pengingat restock kepada pelanggan yang pernah membeli produk tersebut.
3. JUMBO BAG RED RETROSPOT (85099B) -> JUMBO BAG RED RETROSPOT (85099B)
   312/4339 pelanggan; support 7.19%; confidence 49.13% (312/635).
   Interpretasi: pembelian ulang produk pada waktu transaksi berbeda. Dapat menjadi dasar pengingat restock kepada pelanggan yang pernah membeli produk tersebut.
Keterbatasan: observasi satu periode, pelanggan tanpa ID dihapus, pelanggan

## Ekspor hasil

In [32]:
import json
result_dir = Path('hasil')
result_dir.mkdir(exist_ok=True)
patterns.to_csv(result_dir/'pola_temporal.csv',index=False)
singletons.to_csv(result_dir/'pola_singleton.csv',index=False)
audit.to_csv(result_dir/'audit_pembersihan.csv',index=False)
with (result_dir/'sequence_pelanggan.json').open('w',encoding='utf-8') as f:
    json.dump({str(cid):seq for cid,seq in customer_sequences.items()}, f, ensure_ascii=False)
print('Ekspor selesai ke folder',result_dir)


Ekspor selesai ke folder hasil
